In [1]:
using Gridap        
using GridapGmsh
using Gridap.Geometry
using Gridap.TensorValues
using Plots
using LinearAlgebra
using  Gridap.Fields
using  Gridap.CellData
using  Gridap.ReferenceFEs  
using  Gridap.Fields
using Gridap.FESpaces
using NearestNeighbors

In [2]:
using LinearAlgebra
using Distributions     
using Plots

In [3]:
model = GmshDiscreteModel("ThreePointBending.msh")
writevtk(model,"ThreePointBending")

Info    : Reading 'ThreePointBending.msh'...
Info    : 38 entities
Info    : 10805 nodes
Info    : 21456 elements
Info    : Done reading 'ThreePointBending.msh'


3-element Vector{Vector{String}}:
 ["ThreePointBending_0.vtu"]
 ["ThreePointBending_1.vtu"]
 ["ThreePointBending_2.vtu"]

In [4]:
const ν12 = 0.185
const E1 = 23230
const E2 = 36440 
const G = 12350

12350

In [5]:
D_x = 0.01
D_y = 0.01
G0_n = 7e-6
G0_s = 7e-6
τ = 1e-9
R = 0.01
Gc_n = 10.0
Gc_s = 30.0  
Fr_Ang = 30.0
tFr_Ang = tand(Fr_Ang)

0.5773502691896258

In [6]:
θ_Mat = 0.0 #Input the bedding plane angle here
r_nbh = 0.2

0.2

In [7]:
using LinearAlgebra
using Random
using Plots

function generate_elliptical_points(N::Int, a, b, θ) #(this theta controls the direction of points generation)

    R = [cos(θ) -sin(θ); sin(θ) cos(θ)]
    r = sqrt.(rand(N))     
    φ = 2.0 .* π .* rand(N)     
    x = r .* cos.(φ)
    y = r .* sin.(φ)
    pts = [x'; y']        
    scale = Diagonal([a, b])
    ellipse_pts = R' * (scale * pts)
point_objs = [Point(ellipse_pts[1, i], ellipse_pts[2, i]) for i in 1:N]
    return point_objs, φ
end

N = 40
a = 0.01 
b = 0.01 
ellipse_pts,Bond_orient = generate_elliptical_points(N, a, b, 0.0)
origin_point = Point(0.0, 0.0)
cod_final = ellipse_pts
r_x = [val[1] for val in cod_final ]
r_y = [val[2] for val in cod_final ]   
N = length(cod_final)
Dir_Vec = ones(N)
μ_0_pdf = 0.0
r_discr = norm.(cod_final)
C = [ (xi - xj)^2 for xi in r_discr, xj in r_discr ]  
ϵ = 0.5e-10
p0 = fill(1.0 / N, N)
x = [p[1] for p in cod_final]
y = [p[2] for p in cod_final]
# scatter(x, y, legend=false,xlabel="x", ylabel="y")


40-element Vector{Float64}:
  0.0006855916055567449
  0.005914420759088277
 -0.0063656266258766995
 -0.00530220864978335
  0.0028046495562871606
  0.003941796721734843
 -0.004919888185292246
  0.003963190809106672
  0.004892698679254571
  0.0063463380320879096
  0.005165465065429287
  0.0057308747930403395
 -0.0020239930356229048
  ⋮
  0.005707680082597392
 -0.0006868600790347891
 -0.003946317176513206
  0.004815794519158166
 -0.007342099219156044
 -0.0013649695118959722
  0.006746535525283024
  0.0002093643090881513
 -0.005134809655225932
  0.005919214026582334
 -0.004402978072048951
 -0.003369269618588835

In [8]:
function ElasFourthOrderConstTensor_Orthotropic(
    E1, E2, ν12, G12, PlanarState
)
    ν21 = ν12 * E2 / E1

    if PlanarState == 1  # Plane stress
        denom = 1 - ν12*ν21

        C1111 = E1 / denom
        C2222 = E2 / denom
        C1122 = ν12 * E2 / denom
        C1112 = 0.0
        C2212 = 0.0
        C1212 = G12

    elseif PlanarState == 2
        denom = (1 - ν12*ν21)

        C1111 = E1*(1 - ν21) / denom
        C2222 = E2*(1 - ν12) / denom
        C1122 = ν12 * E2 / denom
        C1112 = 0.0
        C2212 = 0.0
        C1212 = G12

    else
        error("PlanarState must be 1 (plane stress) or 2 (plane strain)")
    end

    return SymFourthOrderTensorValue(
        C1111, C1112, C1122,
        C1112, C1212, C2212,
        C1122, C2212, C2222
    )
end

C_mat_local = ElasFourthOrderConstTensor_Orthotropic(E1, E2,ν12,G ,2)

SymFourthOrderTensorValue{2, Float64, 9}(17424.052605393455, 0.0, 7123.861833873065, 0.0, 12350.0, 0.0, 7123.861833873065, 0.0, 31383.499430305663)

In [9]:
function rotation_matrix_2D(θ)
    c = cosd(θ)
    s = sind(θ)
    return  [ c  -s
                      s   c ]
end

function rotate_stiffness_tensor(Cmat, θ)
    Q = rotation_matrix_2D(θ)

    # Allocate rotated tensor
    Crot = zeros(Float64, 2,2,2,2)

    for i in 1:2, j in 1:2, k in 1:2, l in 1:2
        for p in 1:2, q in 1:2, r in 1:2, s in 1:2
            Crot[i,j,k,l] += Q[i,p]*Q[j,q]*Q[k,r]*Q[l,s]*Cmat[p,q,r,s]
        end
    end

    return SymFourthOrderTensorValue(
        Crot[1,1,1,1], Crot[1,1,1,2], Crot[1,1,2,2],
        Crot[1,1,1,2], Crot[1,2,1,2], Crot[2,2,1,2],
        Crot[1,1,2,2], Crot[2,2,1,2], Crot[2,2,2,2]
    )
end
C_mat = rotate_stiffness_tensor(C_mat_local, θ_Mat)


SymFourthOrderTensorValue{2, Float64, 9}(17424.052605393455, 0.0, 7123.861833873065, 0.0, 12350.0, 0.0, 7123.861833873065, 0.0, 31383.499430305663)

In [10]:
function σ(ε)
    σ_elas = (C_mat⊙ε)
    return  σ_elas
end

σ (generic function with 1 method)

In [11]:
degree = 2
Ω = Triangulation(model)
dΩ = Measure(Ω,degree)

GenericMeasure()

In [12]:
labels = get_face_labeling(model)

FaceLabeling:
 0-faces: 10805
 1-faces: 32237
 2-faces: 21433
 tags: 4
 entities: 38

In [13]:
Gr = get_grid(model)
nodes = get_node_coordinates(Gr)
Nₑ, Nₙ = num_cells(model), num_nodes(model)
nodeCoordX, nodeCoordY = [nodes[i][1] for i in 1:Nₙ], [nodes[i][2] for i in 1:Nₙ]
elem = get_cell_node_ids(Gr)

21433-element Gridap.Arrays.Table{Int32, Vector{Int32}, Vector{Int32}}:
 [3086, 6172, 8045]
 [6073, 6074, 10120]
 [4256, 4259, 10360]
 [4567, 6010, 10230]
 [3744, 4545, 6401]
 [4206, 6070, 9679]
 [4206, 6070, 9680]
 [222, 6341, 6418]
 [206, 6556, 10203]
 [6103, 10056, 10467]
 [5494, 5497, 6430]
 [195, 10101, 10203]
 [5109, 6230, 9973]
 ⋮
 [4334, 10766, 10776]
 [306, 5476, 10319]
 [3836, 9993, 10027]
 [4567, 4568, 10230]
 [4778, 10001, 10430]
 [5108, 5109, 9973]
 [5110, 9641, 9931]
 [6017, 9971, 10000]
 [6096, 9751, 10525]
 [6127, 6128, 10256]
 [6378, 10373, 10587]
 [9727, 9729, 10004]

In [14]:
using NearestNeighbors
data = zeros(2,Nₙ)
data[1,:] =nodeCoordX
data[2,:] =nodeCoordY
points = data
balltree = BallTree(data)
idxs = inrange(balltree, points, r_nbh, true)

10805-element Vector{Vector{Int64}}:
 [1]
 [2]
 [3]
 [4, 55, 56, 57, 58, 176, 185, 3536, 3537, 5314  …  9692, 9707, 9750, 9930, 10028, 10215, 10347, 10366, 10573, 10774]
 [5]
 [6]
 [7]
 [8]
 [9]
 [10]
 [11, 129, 130, 131, 177, 5956, 6508, 6571, 6572, 10328, 10429]
 [12, 142, 143, 144, 178, 6064, 6389, 6509, 6573, 10186, 10414, 10779]
 [13]
 ⋮
 [2034, 3889, 3892, 3893, 3894, 5391, 5445, 6091, 6092, 6241, 10107, 10219, 10380, 10710, 10794]
 [10795]
 [6197, 6198, 6217, 9906, 9924, 9925, 10305, 10796]
 [2870, 2871, 3741, 3742, 3743, 3744, 3774, 4477, 4532, 4542  …  6517, 6565, 6566, 6567, 7826, 10072, 10114, 10711, 10788, 10797]
 [3081, 3083, 3084, 3085, 3110, 3111, 3296, 3302, 4434, 4621  …  6410, 6481, 7191, 9780, 9962, 10178, 10402, 10445, 10626, 10798]
 [1370, 3127, 3172, 5887, 5888, 5889, 5890, 5891, 5892, 6015  …  7733, 9959, 10073, 10279, 10451, 10459, 10546, 10698, 10772, 10799]
 [327, 328, 329, 330, 331, 332, 4427, 5461, 5501, 5881, 6202, 6439, 6980, 8780, 9819, 10299, 10475, 1065

In [15]:
function G_kill(σ_eq_x_n, G0_eq)
G_kill_x = 0.5 .* G0_eq .* ((σ_eq_x_n  .- 1) .+ abs.(σ_eq_x_n  .- 1)  ) .* σ_eq_x_n ./ T
    return G_kill_x
end 

G_kill (generic function with 1 method)

In [16]:
function new_EnergyState(ψPlusPrev_in,ψhPos_in)
    ψPlus_in = ψhPos_in
    if ψPlus_in <= ψPlusPrev_in
        ψPlus_out = ψPlusPrev_in 
        damaged = false
    else
        ψPlus_out = ψPlus_in
        damaged = true
    end
    damaged, ψPlus_out   
end

new_EnergyState (generic function with 1 method)

In [17]:
maxiter = 1
sinkhorn_iters = 1
tol = 1e-9
function sinkhorn_JKO_step(p_k_x, C, ϵ, τ, V_x, sinkhorn_iters, tol)

    N = length(p_k_x)
    p_k_x = Float64.(p_k_x)
   
    K =  exp.(-Float64.(C) ./ ϵ) 
    p_next_x = copy(p_k_x)
    q_x = copy(p_k_x)

    for t in 1:maxiter
        q_x = exp.(-2*(τ/ϵ) .* (2 ./ r_discr) .* V_x  .* D_x)

        p_tilde_x = p_k_x .* q_x

        u_x = ones(N)
        v_x = ones(N)
      
        small = 1e-16
        for _ in 1:sinkhorn_iters
            u_x = p_k_x ./ (K * v_x .+ small)
            v_x = p_tilde_x ./ (K' * u_x .+ small)

        end

        Π_x = Diagonal(u_x) * K * Diagonal(v_x)
        p_new_x = vec(sum(Π_x, dims=1))
        p_next_x = p_new_x

    end
    bar_w_nds_x = sum(p_next_x .* exp.(dot.(-cod_final, cod_final)))

    return bar_w_nds_x  ,  p_next_x
end

sinkhorn_JKO_step (generic function with 1 method)

In [18]:
function nonLocal_w_bar_fe(ε_field, p_0_nds_x,Gk_history_old) 
    ε_vals = evaluate(ε_field, x_S) 
    
    caches_x = [array_cache(ε_vals) for _ in 1:Threads.nthreads()]
    T_type = eltype(ε_vals[1]) 
    
    Gk_sum_x = zeros(T_type, Nₙ)
    Gk_count_x = zeros(Int, Nₙ)
    
    for iel in 1:Nₑ 
        cache_x = caches_x[Threads.threadid()]
        ElNdInd = elem[iel]
        val_ε_tensor = getindex!(cache_x, ε_vals, iel)
        tensor_ip = (sum(val_ε_tensor))/(length(val_ε_tensor)) 
        for node in ElNdInd
            Gk_sum_x[node] += tensor_ip
            Gk_count_x[node] += 1
        end
    end
    Gk_nds_x = Gk_sum_x ./ Gk_count_x
  
    w_bar_nds_x = zeros(Nₙ)
    p_new_x = zeros(N, Nₙ) 
    
    Gk_history_new = zeros(N, Nₙ)
    
    Threads.@threads for nd_id in 1:Nₙ
       
        G_tensor_nd = Gk_nds_x[nd_id]
        Gk_nd_Aniso_x1_temp = map(r -> dot(r, G_tensor_nd ⋅ r), cod_final ./ norm.(cod_final))
          Gk_nd_Aniso_x2_temp = map(n -> begin
    n_perp = VectorValue(-n[2], n[1]) 
    dot(n_perp, G_tensor_nd ⋅ n_perp)
            end, cod_final ./ norm.(cod_final)) 
        
        Gk_nd_Aniso_rn_temp = map(r -> begin
 n_perp = VectorValue(-r[2], r[1]) 
    dot(r, G_tensor_nd ⋅ n_perp) + dot(n_perp, G_tensor_nd ⋅ r)
end, cod_final ./ norm.(cod_final))
        
        Gk_nd_Aniso_x2 = Gk_nd_Aniso_x2_temp
        Gk_nd_Aniso_x1 = Gk_nd_Aniso_x1_temp 
        
        Gk_nd_Aniso_x1_nor = max.(Gk_nd_Aniso_x2_temp,Gk_nd_Aniso_x1_temp,0.0)
        Gk_nd_Aniso_x1_shear = 0.5 .* Gk_nd_Aniso_rn_temp
        Gk_nd_Aniso_x1_comp = min.(Gk_nd_Aniso_x2_temp + Gk_nd_Aniso_x1_temp,0.0)
        Gk_nd_Aniso_x1_cs = min.(Gk_nd_Aniso_rn_temp,0.0)
        
        Gk_nd_Eq = sqrt.( Gk_nd_Aniso_x1_nor .^2 ./ Gc_n.^2 +  Gk_nd_Aniso_x1_shear.^2 ./((Gc_s .- Gk_nd_Aniso_x1_nor .* tFr_Ang .* sign.(Gk_nd_Aniso_x1_shear) ).^2))
        mod_mix =  atan.(( max.(Gk_nd_Aniso_x2_temp, Gk_nd_Aniso_x1_temp) ) ./ abs.( ( (Gk_nd_Aniso_x1_shear)  ) .+ 1e-8) )
         G0_eq = sqrt.( (G0_n.^2 * G0_s.^2) ./ (G0_n.^2 .* (cos.(mod_mix)).^2 .+ G0_s.^2 .* (sin.(mod_mix)).^2  ))
        Gk_nd_Aniso = G_kill(Gk_nd_Eq , G0_eq)
        
        Gk_old_local = Gk_history_old[:, nd_id]
        Gk_max_local = max.(Gk_nd_Aniso, Gk_old_local)
        Gk_history_new[:, nd_id] .= Gk_max_local
        p_0_in_x = p_0_nds_x[:, nd_id]
        
        @inbounds w_bar_nds_x[nd_id], p_temp_x = sinkhorn_JKO_step(
            p_0_in_x, C, ϵ, τ,  Gk_max_local, sinkhorn_iters, tol
        )
        p_new_x[:, nd_id] .= p_temp_x
    end

    return FEFunction(Vphi, w_bar_nds_x .+ 1e-6), p_new_x, Gk_history_new 
end

nonLocal_w_bar_fe (generic function with 1 method)

In [19]:
LoadTagId = get_tag_from_name(labels,"LoadEdge")
Γ_Load = BoundaryTriangulation(model,tags = LoadTagId)
dΓ_Load = Measure(Γ_Load,degree)
n_Γ_Load = get_normal_vector(Γ_Load)

GenericCellField():
 num_cells: 15
 DomainStyle: ReferenceDomain()
 Triangulation: BoundaryTriangulation()
 Triangulation id: 1702588249129236384

In [20]:
reffe_Disp = ReferenceFE(lagrangian,VectorValue{2,Float64},1)
V0_Disp = TestFESpace(model,reffe_Disp;
          conformity=:H1,
          dirichlet_tags=["LeftEdge","RightEdge","LoadEdge"],
          dirichlet_masks=[(false,true),(true,true),(false,true)])

UnconstrainedFESpace()

In [21]:
reffephi  = ReferenceFE(lagrangian,Float64,1)
Vphi  = TestFESpace(model,reffephi;
          conformity=:H1)

UnconstrainedFESpace()

In [22]:
function step_disp(uApp,uh,p_0_nds_x,ϕ,Gk_history_new)
uApp1(x) = VectorValue(0.0,0.0)
uApp2(x) = VectorValue(0.0,0.0)  
uApp3(x) = VectorValue(0.0,-uApp)
U_Disp = TrialFESpace(V0_Disp,[uApp1,uApp2,uApp3])
ϕ,p_0_nds_x,Gk_history_new  = nonLocal_w_bar_fe(σ∘((ε(uh))) ,p_0_nds_x,Gk_history_new)
a(u,v) = ∫( ε(v) ⊙ ((σ∘((ε(u))))  .*(ϕ+1e-6)))dΩ
l(v) = 0.0 
op = AffineFEOperator(a,l,U_Disp,V0_Disp)
ls = LUSolver()
solver = LinearFESolver(ls)
uh = solve(solver,op)
    return uh, ϕ, p_0_nds_x, Gk_history_new 
end

step_disp (generic function with 1 method)

In [23]:
function project(q,model,dΩ,order)
  reffe = ReferenceFE(lagrangian,Float64,order)
  V = FESpace(model,reffe,conformity=:L2)
  a(u,v) = ∫( u*v )*dΩ
  l(v) = ∫( v*q )*dΩ
  op = AffineFEOperator(a,l,V,V)
  qh = solve(op)
  qh
end

project (generic function with 1 method)

In [24]:
dΩ_ro = Measure(Ω,1)
x_S = get_cell_points(dΩ_ro)

CellPoint()

In [25]:
cd("3PB_ortho_results")

In [27]:
uh = zero(V0_Disp)
Tmax = 0.006
delT = 2e-5
vApp = 1.0
count_n = 0
T = 0.0
Load = Float64[]
Displacement = Float64[]
push!(Load, 0.0)
push!(Displacement, 0.0)
Gk_history_new = zeros(N, Nₙ)
uh_prev = zero(V0_Disp)
uh_in_FE = uh
f_new = 1.0
ϕ_prev = interpolate_everywhere(f_new,Vphi)
ϕ = interpolate_everywhere(f_new,Vphi)
ϕ_x = interpolate_everywhere(f_new,Vphi)
ϕ_y = interpolate_everywhere(f_new,Vphi)
innerMax = 10
G_k_cell_x = CellState(0.0,dΩ_ro)
G_k_cell_y = CellState(0.0,dΩ_ro)
p0_nds_x = p0 .* ones(1, Nₙ)
p0_nds_y = p0 .* ones(1, Nₙ)
start_time = time() 
while T <= Tmax
    count_n = count_n + 1
T = T + delT
uApp  = T*vApp
    print("\n Entering displacemtent step :", float(uApp))
 for inner = 1:innerMax
uh,ϕ,p0_nds_x,Gk_history_new =  step_disp(uApp,uh,p0_nds_x,ϕ,Gk_history_new)
e = uh - uh_in_FE

err = sqrt(sum( ∫( e⊙e )*dΩ ))
ϕ_prev = ϕ
uh_in_FE = uh
print("\n error = ",float(err))
        if err < 1e-8
            break 
        end  
    end
Node_Force = sum(∫( n_Γ_Load ⋅ (σ∘( (ε(uh))) ).*ϕ)dΓ_Load)
push!(Load, abs(Node_Force[2]))
push!(Displacement, uApp)  
if mod(count_n,10) == 0
writevtk(Ω,"NDB_sinkhorn_results_$count_n",cellfields=["disp"=>uh,"phi"=>ϕ, "sig" => σ∘( (ε(uh))) ])   
    end
    
    end
end_time = time()
elapsed_time = end_time - start_time


 Entering displacemtent step :2.0e-5
 error = 0.0005905129692309202
 error = 3.3550255200513307e-16
 Entering displacemtent step :4.0e-5
 error = 0.0005905129692315679
 error = 7.996510214601974e-16
 Entering displacemtent step :6.000000000000001e-5
 error = 0.0005905129692297196
 error = 5.470101251852343e-16
 Entering displacemtent step :8.0e-5
 error = 0.0005905129692314187
 error = 2.637455413233724e-15
 Entering displacemtent step :0.0001
 error = 0.0005905129692337457
 error = 6.277529850094622e-16
 Entering displacemtent step :0.00012
 error = 0.000590512969229374
 error = 2.562607426105261e-15
 Entering displacemtent step :0.00014000000000000001
 error = 0.0005905129692303854
 error = 1.2872398670454037e-15
 Entering displacemtent step :0.00016
 error = 0.0005905129692313044
 error = 6.291255923919413e-16
 Entering displacemtent step :0.00018
 error = 0.0005905129692311708
 error = 3.206757690360914e-15
 Entering displacemtent step :0.0002
 error = 0.0005905129692352458
 error

2245.6819999217987